In [1]:
# external
import numpy as np
import healpy as hp
import time
from tqdm import trange
# modules from cmblensplus
import cmblensplus.curvedsky as cs
import constant as c
import cmb

ModuleNotFoundError: No module named 'cmblensplus.curvedsky._core'

In [ ]:
import cmblensplus
print(cmblensplus.__file__)

In [ ]:
def benchmark(func, niter, desc):
    times = []

    for _ in trange(niter, desc=desc):
        t0 = time.perf_counter()
        func()
        times.append(time.perf_counter() - t0)

    times = np.asarray(times)

    print(f"{desc}")
    print(f"median: {np.median(times)}")
    print(f"mean:   {np.mean(times)}")
    print(f"min:    {np.min(times)}")


In [ ]:
niter = 10

In [ ]:
Lmax = 2*2048
# ucl is an array of shape [0:5,0:rlmax+1] and ucl[0,:] = TT, ucl[1,:] = EE, ucl[2,:] = TE, lcl[3,:] = phiphi, lcl[4,:] = Tphi
ucl = cmb.read_camb_cls('../data/unlensedcls.dat',ftype='scal',output='array')[:,:Lmax+1]

Generate random gaussian fields

In [ ]:
Talm = cs.utils.gauss1alm(ucl[0,:])

In [ ]:
alm = hp.synalm(ucl[0,:], lmax=Lmax, new=True)

Set parameters

In [ ]:
nside = 1024  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside

In [ ]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [ ]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [ ]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

In [ ]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

In [ ]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

In [ ]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)

Higher resolution

In [ ]:
nside = 2048  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside # Maximum multipole of harmonic coefficients to be computed

In [ ]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [ ]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [ ]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

In [ ]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

In [ ]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

In [ ]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)